In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

In [2]:
benchmark_df = pd.read_csv(r'./uniprot_dataset/benchmark_df.csv')
reproduced_df = pd.read_csv(r'./uniprot_dataset/reproduced_df.csv')

In [4]:
print(len(benchmark_df))
print(len(reproduced_df))

74731
72302


In [7]:
# given: source is a superset of target
# output: intersection
def get_merge_entries(source, target, source_col, target_col):
    merge_entries = source.rename(
        columns = dict(zip(source_col, target_col))
    ).merge(
        target, on = target_col, how='left', indicator=True
    )
    
    missing_entries = merge_entries[merge_entries['_merge'] == 'left_only'].drop(columns=['_merge'])
    print(f"Missing (seq_id, mut_name) pairs in target: {len(missing_entries)}")
    print(missing_entries.head())

    return merge_entries

In [8]:
merge_entries = get_merge_entries(benchmark_df, reproduced_df, ['uniprot_id', 'aa_change'], ['seq_id', 'mut_name'])
benchmark_df = merge_entries[merge_entries['_merge'] == 'both'][['seq_id', 'mut_name', 'ESM1b_score', 'clinvar_label']].rename(
    columns={'seq_id' : 'uniprot_id', 'mut_name' : 'aa_change'}
)
benchmark_df.head()

Missing (seq_id, mut_name) pairs in target: 2429
     seq_id mut_name  ESM1b_score  clinvar_label  esm_score
14   A0JNW5    S797L       -4.651            NaN        NaN
62   A1L188      M1V       -7.404            1.0        NaN
63   A1L390    R958H       -5.545            NaN        NaN
126  A2RRH5    L133P       -1.776            NaN        NaN
177  A2RU37     V84I       -3.275            NaN        NaN


,uniprot_id,aa_change,ESM1b_score,clinvar_label
0,A0AUZ9,N660S,-3.064,NaN
1,A0AV02,R664Q,-7.572,NaN
2,A0AV02,K342R,-5.013,NaN
3,A0AV02,R181C,-10.698,NaN
4,A0AV02,K541R,-2.938,NaN


In [9]:
benchmark_score = benchmark_df['ESM1b_score'].values
reproduced_score_dict = {'ESM1b' : reproduced_df['ESM1b_score'].values,
                         'ESM1v-1' : reproduced_df['ESM1v-1_score'].values,
                         'ESM2' : reproduced_df['ESM2_score'].values
                        }

KeyError: 'ESM1b_score'

In [ ]:
differences = np.array(benchmark_score) - np.array(reproduced_score_dict['ESM1b'])

rms = np.sqrt(np.mean(differences**2))
print(f"RMS of differences: {rms}")

significant_indices = np.where(abs(differences) > 5)[0].tolist()
significant_names = reproduced_df.iloc[significant_indices][['seq_id', 'mut_name']].to_dict(orient='records')
significant_annotations = dict(zip(significant_indices, significant_names))
print(f'indices of mutations with diff > 5: {significant_annotations}')
print(f'Total = {len(significant_annotations)}')

plt.figure(figsize=(10, 5))

plt.plot(differences, marker='o', linestyle='-', color='purple', alpha=0.7)

plt.axhline(0, color='black', linestyle='dashed', linewidth=2)

plt.xlabel('Index')
plt.ylabel('Difference (Benchmark - Reproduced)')
plt.title('ESM1b Score Differences')

plt.savefig(r'./uniprot_dataset/difference_plot.jpg')
plt.show()

In [ ]:
benchmark_df_labeled = benchmark_df.dropna(subset=['clinvar_label'])
merge_entries = get_merge_entries(benchmark_df, reproduced_df, ['uniprot_id', 'aa_change'], ['seq_id', 'mut_name'])

reproduced_df_labeled = merge_entries[merge_entries['_merge'] == 'both'][['seq_id', 'mut_name', 'ESM1b_score',
                                                                          'ESM1v-1_score', 'ESM2_score', 'clinvar_label']]

benchmark_score_labeled = -benchmark_df_labeled['ESM1b_score'].values
benchmark_label = benchmark_df_labeled['clinvar_label'].values
reproduced_score_labeled_dict = {'ESM1b' : -reproduced_df['ESM1b_score'].values,
                                 'ESM1v-1' : -reproduced_df['ESM1v-1_score'].values,
                                 'ESM2' : -reproduced_df['ESM2_score'].values
                                }
reproduced_label = reproduced_df_labeled['clinvar_label'].values

In [ ]:
benchmark_roc_auc = roc_auc_score(benchmark_label, benchmark_score_labeled)
reproduced_roc_auc = {model_name : roc_auc_score(reproduced_label, model_value) for model_name, model_value in reproduced_score_labeled_dict.items()}

print(f'ROC_AUC for benchmark = {benchmark_roc_auc}')
print(f'ROC_AUC for reproduced = {reproduced_roc_auc}')